# Notebook 03 — Vina Baseline Scoring

**Goals:**
1. Prepare receptor PDBQT files (convert PDB → PDBQT with partial charges)
2. Dock 3 clinical drugs (Erlotinib / Gefitinib / Osimertinib) on both targets
3. Build the baseline Vina score table for later comparison

**Why this matters:** when we later generate 1000 molecules with TargetDiff, we need to compare their Vina scores against something. These 3 FDA-approved EGFR drugs are the benchmark — a generated molecule scoring better than Erlotinib is a strong positive signal.

**Inputs (from notebook 02):**
- `data/pdb/1M17.pdb`, `data/pdb/4I22.pdb`
- `data/pocket_centers.json`
- `data/baselines/*.sdf`

**Outputs saved to Drive:**
- `data/pdb/1M17_receptor.pdbqt`, `data/pdb/4I22_receptor.pdbqt`
- `results/vina_scores/baselines.csv` (6 rows: 3 drugs × 2 targets)

Runtime: **CPU**, total ~5 minutes (Vina docking is CPU-parallel).


In [ ]:
# ── Colab bootstrap ─────────────────────────────────────────────────────────
REPO_URL = "https://github.com/Jonathan-Ye-1/egfr-drug-design-eecs6895-final-project.git"

import os, sys, subprocess
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

PROJECT_ROOT = '/content/drive/MyDrive/egfr_diffusion'
if not os.path.exists(os.path.join(PROJECT_ROOT, '.git')):
    subprocess.run(['git', 'clone', REPO_URL, PROJECT_ROOT], check=True)
else:
    subprocess.run(['git', '-C', PROJECT_ROOT, 'pull'], check=False)
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)

from src.colab_init import setup
PROJECT_ROOT = setup(repo_url=REPO_URL)

# ── Install AutoDock Vina binary + confirm obabel ───────────────────────────
subprocess.run(['pip', 'install', '-q', 'vina'], check=False)
subprocess.run(['apt-get', 'install', '-y', '-qq', 'openbabel', 'autodock-vina'], check=False)
print('vina:', subprocess.run(['which', 'vina'], capture_output=True, text=True).stdout.strip())
print('obabel:', subprocess.run(['which', 'obabel'], capture_output=True, text=True).stdout.strip())

# ── Load data from notebook 02 ──────────────────────────────────────────────
import json
import pandas as pd
from src.utils import load_config
from src.docking import VinaDocker, dock_molecules

cfg = load_config(f'{PROJECT_ROOT}/configs/targets.yaml')
with open(f'{PROJECT_ROOT}/data/pocket_centers.json') as f:
    centers = json.load(f)
print('Pocket centers:', centers)


In [ ]:
# ── Prepare receptor PDBQT ──────────────────────────────────────────────────
# Requires obabel (Open Babel) installed:
#   conda install -c conda-forge openbabel
#   OR: sudo apt install openbabel  (Colab/Linux)

from src.pocket_extraction import prepare_receptor_pdbqt

for pdb_id, target_key in [('1M17', 'wildtype'), ('4I22', 'mutant')]:
    pdb_file   = f'{PROJECT_ROOT}/data/pdb/{pdb_id}.pdb'
    pdbqt_file = f'{PROJECT_ROOT}/data/pdb/{pdb_id}_receptor.pdbqt'
    if not os.path.exists(pdbqt_file):
        prepare_receptor_pdbqt(pdb_file, pdbqt_file)
    else:
        print(f'{pdb_id} PDBQT already prepared.')

In [ ]:
# ── Dock baseline drugs on both pockets ────────────────────────────────────
# Expected: Erlotinib on 1M17 around -9 to -11 kcal/mol
from rdkit import Chem
import os

os.makedirs(f'{PROJECT_ROOT}/results/vina_scores', exist_ok=True)

results = []
for pdb_id in ['1M17', '4I22']:
    center = centers[pdb_id]
    docker = VinaDocker(
        receptor_pdbqt=f'{PROJECT_ROOT}/data/pdb/{pdb_id}_receptor.pdbqt',
        center=tuple(center),
        box_size=(20.0, 20.0, 20.0),
        exhaustiveness=16,
    )
    for drug_name, info in cfg['baselines'].items():
        sdf_path = os.path.join(PROJECT_ROOT, info['sdf_file'])
        mol = next(Chem.SDMolSupplier(sdf_path))
        score = docker.dock_mol(mol)
        results.append({'target': pdb_id, 'drug': info['name'], 'vina_score': score})
        print(f'{pdb_id} / {info["name"]}: {score}')

baseline_df = pd.DataFrame(results)
baseline_df.to_csv(f'{PROJECT_ROOT}/results/vina_scores/baselines.csv', index=False)
print()
print('=== Baseline Vina Score Matrix (kcal/mol, lower = better) ===')
print(baseline_df.pivot(index='drug', columns='target', values='vina_score'))


In [ ]:
# ── Erlotinib re-dock RMSD check ────────────────────────────────────────────
# NOTE: Computing RMSD requires the crystal pose PDBQT output from Vina.
# A score close to the literature (Erlotinib ~-9 to -11 on 1M17) is a good sign.
# For RMSD: compare Vina output pose vs. crystal ligand with ProDy or RDKit.

erlotinib_score_wt = baseline_df.query('drug=="Erlotinib" and target=="1M17"')['vina_score'].values
if len(erlotinib_score_wt):
    print(f'Erlotinib on 1M17 (WT): {erlotinib_score_wt[0]:.2f} kcal/mol')
    if erlotinib_score_wt[0] > -7.0:
        print('⚠ Score unusually weak — check receptor prep and box definition.')
    else:
        print('✓ Score looks reasonable.')